# Arquitectura Medallion aplicada a CAPTIA Synthetic Data BMS

> _Tutorial · Caso de uso: **Overview** · Capa Medallion: **transversal** · Spec: `docs/specs/synthetic-bms/01-product-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Entender en 30 minutos cómo se organizan los datos del proyecto en las tres capas Medallion (bronce → plata → oro), por qué CAPTIA tiene una capa plata canónica (`captia_point` + 5 tags + `value`) y cómo encaja cada caso de uso del proyecto en esta arquitectura.


## 2. Qué se aprende

- Definición precisa de cada capa Medallion.
- Diferencia entre Medallion estricto, distribuido e híbrido.
- Por qué el InfluxDB de simarro-prod ya es **una capa plata**.
- Cómo cada caso de uso (A–J) se proyecta en bronce / plata / oro.
- Vocabulario que usaremos en todos los demás notebooks.


## 3. Contexto del caso de uso

El proyecto del Curso de Especialización IA & Big Data del IES Simarro trabaja con datos de edificios inteligentes. La estrategia de datos adoptada (mayo 2026) es **Medallion híbrida**: cada equipo construye una capa plata local con el schema canónico CAPTIA, lo que permite trabajar en paralelo sin bloqueos y consolidar al final del proyecto sin reescribir el código.


## 4. Relación con CENTINELA+

CENTINELA+ ya tiene resuelto el paso bronce → plata para sensores reales: los sensores publican payloads MQTT que Telegraf normaliza y escribe en InfluxDB con el schema canónico. Cuando los datos del IES Simarro estén disponibles, los modelos entrenados sobre nuestra capa plata sintética deben cambiar **solo la URL y el token** del cliente InfluxDB (no el código).


## 5. Relación con Medallion

Diagrama:

```mermaid
flowchart LR
  B[CAPA BRONCE\nCSV / NetCDF / JPEG] --> S[CAPA PLATA\ncaptia_point + 5 tags + value]
  S --> O1[ORO Caso B\nfeatures forecast]
  S --> O2[ORO Caso C\nIF/AE + labels]
  S --> O3[ORO Caso D\npivot IAQ + occupancy]
  S --> O4[ORO Caso H\ntools + RAG]
```


## 6. Datos de entrada

Este notebook es **conceptual**: no carga ningún dataset. Visualizaremos tablas comparando capas y un Mermaid resumen de los 11 casos.


## 7. Schema CAPTIA esperado

Aunque no hagamos ETL, fijamos las constantes que aparecerán en todos los notebooks siguientes.


In [ ]:
print("MEASUREMENT:", MEASUREMENT_TELEMETRY)
print("CANONICAL TAGS:", CANONICAL_TAGS)
print("BUCKETS:", list(DEFAULT_BUCKET_RETENTIONS.keys()))


## 8. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.


In [ ]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


## 9. Carga de datos o mock

Construimos en memoria una tabla resumen con los 11 casos para guiar el resto del proyecto. No hay descargas externas.


In [ ]:
casos = pd.DataFrame(
    [
        ("A", "Pipeline IoT", "MQTT → Telegraf → InfluxDB", "telemetry", "dashboards Grafana"),
        ("B", "Forecast consumo 24h", "BDG2 + UCI", "bms_buildings + bms_classrooms", "features + SARIMA/XGB"),
        ("C", "Anomalías HVAC", "LBNL FDD + sintético", "captia_fault_labels", "Isolation Forest + AE"),
        ("D", "IAQ + ocupación", "In-Gauge + UCI Occ", "telemetry 1m", "Random Forest"),
        ("E", "Meteorología solar", "ERA5 + AEMET", "weather_station/xativa", "predictor solar"),
        ("F", "MLOps", "(transversal)", "(transversal)", "MLflow + lakeFS"),
        ("G", "Calidad con agentes", "GE + reglas Flux", "todas las capas", "agentes evaluadores"),
        ("H", "RAG + Chatbot", "InfluxDB + docs", "(consume plata)", "tools + RAG"),
        ("I", "Spark vs Pandas", "BDG2 53M filas", "subset → Caso B", "benchmark"),
        ("J", "Tráfico YOLO", "JPEG DGT + AEMET", "traffic_cameras", "predicción congestión"),
        ("Extra", "Test calidad chatbot", "golden set", "(consume H)", "scoring"),
    ],
    columns=["caso", "objetivo", "bronce", "plata", "oro"],
)
casos


## 10. Exploración paso a paso

Comparamos las características de las tres capas en una tabla.


In [ ]:
capas = pd.DataFrame(
    {
        "Bronce": [
            "datos crudos en su formato original",
            "puede tener nulos, duplicados, unidades distintas",
            "se versiona; nunca se modifica",
        ],
        "Plata": [
            "schema canónico captia_point",
            "5 tags + field value, unidades SI",
            "alimentada por ETL; consumida por modelos",
        ],
        "Oro": [
            "datasets específicos por caso de uso",
            "features ML, embeddings, agregaciones",
            "evolutiva; vive cerca del modelo",
        ],
    },
    index=["¿Qué contiene?", "Reglas", "Quién la usa"],
)
capas


## 11. Transformación bronce → plata

Un fragmento muestra cómo construiríamos una línea de line-protocol desde un sensor mock; reaparecerá en notebooks operacionales.


In [ ]:
ts_ns = int(pd.Timestamp("2026-05-10T12:00:00Z").value)
linea = build_line_protocol(
    measurement=MEASUREMENT_TELEMETRY,
    tags={
        "captia_env": "dev",
        "domain_id": "bms_classrooms",
        "site_id": "ies_simarro",
        "asset_id": "AULA01",
        "variable": "co2",
    },
    fields={"value": 712.5},
    timestamp_ns=ts_ns,
)
print(linea)


## 12. Construcción de capa oro

La capa oro es **caso-específica**: features ML para forecasting, embeddings para RAG, datasets etiquetados para detección de anomalías. No la construimos aquí — cada caso le dedica un notebook propio.


## 13. Visualizaciones explicativas

Visualizamos cuántos notebooks tiene cada caso (mostraremos el plan de ejecución del proyecto).


In [ ]:
counts = pd.Series(
    {
        "00 Overview": 3, "A Pipeline IoT": 3, "B Forecast": 5,
        "C Anomalías": 5, "D IAQ": 5, "E Meteo": 4,
        "F MLOps": 3, "G Calidad": 4, "H RAG": 5,
        "I Spark": 4, "J Tráfico": 4,
    },
    name="notebooks por caso",
)
ax = counts.sort_values().plot.barh(figsize=(8, 4), color="#3F51B5")
ax.set_xlabel("notebooks")
ax.set_title("Plan didáctico — 42 notebooks")
plt.tight_layout()


## 14. Validaciones

Comprobamos que las constantes están bien expuestas y que los buckets esperados aparecen.


In [ ]:
assert MEASUREMENT_TELEMETRY == "captia_point"
assert set(CANONICAL_TAGS) == {"captia_env", "domain_id", "site_id", "asset_id", "variable"}
assert "telemetry" in DEFAULT_BUCKET_RETENTIONS
print("Schema canónico OK")


## 15. Errores comunes

1. **Modificar el measurement.** Cambiar `captia_point` rompe Telegraf y los dashboards.
2. **Usar `value_<tipo>`.** Solo existe el field `value` (float).
3. **Cardinalidad alta de tags.** Los 5 tags son indexados; añadir un sexto causa explosión de series.
4. **Mezclar continuos y on-change en un mismo bucket.** Los `_state` van a `state_events`.
5. **Hardcodear tokens.** Siempre usar `.env`.


## 16. Ejercicios propuestos

1. Para cada caso de la tabla `casos`, identifica un dataset público alternativo válido.
2. Construye un line-protocol para `power_01` con valor 850 W.
3. Explica por qué `state_events` tiene 90 días y `telemetry` solo 14.


## 17. Cómo se reutiliza con datos reales

Cuando CAPTIA proporcione un dump real de InfluxDB, el cambio para todos los notebooks es:

1. `influx restore` del dump en el bucket `telemetry`.
2. Actualizar `INFLUXDB_URL` y `INFLUXDB_TOKEN` en `.env`.
3. Re-ejecutar las celdas de query — el resultado pasa de mock a real.


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Siguiente notebook: `00_project_overview/01_schema_captia_influxdb.ipynb`.
- Documento web del caso: `docs/architecture/medallion.md`.
- Recordar `seed=42` y schema canónico al iniciar cualquier notebook.
